<a href="https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [13]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample': f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [14]:
print("dim_content columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['dim_content']} LIMIT 1").df())

print("\nfact_daily columns:")
print(con.sql(f"DESCRIBE SELECT * FROM {TABLES['fact_daily']} LIMIT 1").df())

dim_content columns:
                   column_name column_type null   key default extra
0               client_hash_id     VARCHAR  YES  None    None  None
1              content_hash_id     VARCHAR  YES  None    None  None
2              keyword_hash_id     VARCHAR  YES  None    None  None
3                  url_hash_id     VARCHAR  YES  None    None  None
4           keyword_char_count      BIGINT  YES  None    None  None
5          keyword_token_count      BIGINT  YES  None    None  None
6               url_char_count      BIGINT  YES  None    None  None
7         content_created_date        DATE  YES  None    None  None
8         content_updated_date        DATE  YES  None    None  None
9                 content_type     VARCHAR  YES  None    None  None
10               search_volume      BIGINT  YES  None    None  None
11                 competition      DOUBLE  YES  None    None  None
12           competition_level     VARCHAR  YES  None    None  None
13                         

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one (client, content) pair on one calendar day — a "page-day" — in
fact_content_daily_performance, joined to that content's static metadata in dim_content.

Tables: dim_content (content attributes: type, word count, publish status) joined to
fact_content_daily_performance on client_hash_id + content_hash_id, filtered to the
month=2026-03 partition (a mid-panel month, not the final-month _sample table).

Time window: one calendar month, 2026-03-01 to 2026-03-31.

What I'd rank: a CTR opportunity proxy, not a hard ground-truth label. For each page-day,
opportunity = how far its actual CTR falls below the empirical CTR benchmark for its
position bucket, weighted by impressions. This is a proxy — "worth a human look" — not a
verified fact about why a page underperforms.

Deliberately excluded: unpublished/deleted content (is_published, is_deleted), rows where
gsc_data_available is FALSE (no real signal that day), and anything beyond the report_date
itself — no forward-looking joins into future months.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Label / proxy: CTR opportunity gap — NOT a column in any table. Computed downstream as
(position-bucket benchmark CTR) − (actual CTR), weighted by impressions. This is a proxy for
"worth reviewing," not a verified ground-truth outcome.

Feature (candidates — knowable at the decision moment, before any outcome): content_type,
main_intent, word_count, char_count, search_volume, competition, backlinks, category_count,
gsc_avg_position, gsc_impressions.

Context (identifiers — for joining/grouping only, never for the model to learn from):
client_hash_id, content_hash_id, keyword_hash_id, url_hash_id, report_date.

Excluded:
- gsc_clicks — this is one of the two raw ingredients CTR itself is computed from
  (CTR = clicks / impressions), and CTR is what my proxy label is built on. Using it as a
  feature would mean predicting the label from a piece of the label. This is exactly the
  trap I'll demonstrate deliberately in Section 3.
- gsc_sum_position — redundant with gsc_avg_position (sum ÷ impressions = avg); keeping both
  adds no information, just doubles up the same signal.
- is_published, is_deleted — product-decision flags, not predictive features. Used only as
  eligibility filters before any modeling happens, never fed to a model.
- Anything from months other than 2026-03 — this contract covers one fixed window; no
  forward-looking

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [15]:
# 1. GRAIN: one row per (client, content, day)? Should return 0 rows.
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) c
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain check (should be empty):")
print(grain_check)

# 2. ROW COUNT + DATE SPAN for the slice
counts = con.sql(f"""
    SELECT COUNT(*) AS rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print("\nSlice size and date span:")
print(counts)

# 3. AVAILABILITY: how many rows actually have real GSC data that day?
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows
    FROM {TABLES['fact_daily']}
    WHERE month = '2026-03'
""").df()
print("\nAvailability (IS TRUE filter):")
print(availability)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (should be empty):
Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []

Slice size and date span:
      rows   min_date   max_date
0  9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability (IS TRUE filter):
   total_rows  gsc_available_rows
0     9841378           3611061.0


Three facts verified against real queries:
- Grain: one row per (client, content, day) confirmed — grain check returned 0 duplicate rows.
- Slice: 9,841,378 rows, spanning the full month (2026-03-01 to 2026-03-31), as expected.
- Availability: only 3,611,061 of 9,841,378 rows (36.7%) have gsc_data_available = TRUE.
  The rest are days with no real GSC signal — must filter on this flag before treating any
  row as a real observation, or CTR/position calculations would be diluted by empty days.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [16]:
features_df = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id, f.report_date,
        d.content_type, d.main_intent, d.word_count, d.search_volume,
        f.gsc_avg_position,
        f.gsc_impressions, f.gsc_clicks
    FROM {TABLES['fact_daily']} f
    JOIN {TABLES['dim_content']} d
      ON f.client_hash_id = d.client_hash_id AND f.content_hash_id = d.content_hash_id
    WHERE f.month = '2026-03'
      AND f.gsc_data_available IS TRUE
      AND d.is_published IS TRUE
      AND d.is_deleted IS FALSE
""").df()

print(f"Feature frame: {len(features_df):,} rows")
print(features_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 3,609,108 rows
            client_hash_id           content_hash_id report_date  \
0  client_2094c6eb080311d5  content_14a3d47ccd0d15dc  2026-03-09   
1  client_2094c6eb080311d5  content_14a6f92117604fef  2026-03-30   
2  client_2094c6eb080311d5  content_14a86c63a214f648  2026-03-19   
3  client_2094c6eb080311d5  content_14b1a02c1b8557fb  2026-03-28   
4  client_2094c6eb080311d5  content_14c17f59aa610ab3  2026-03-14   

      content_type    main_intent  word_count  search_volume  \
0  keyword article  informational        3864              0   
1  keyword article  informational        2867              0   
2  keyword article  informational        2949              0   
3  keyword article  informational        4279              0   
4  keyword article  informational        3277            140   

   gsc_avg_position  gsc_impressions  gsc_clicks  
0          5.000000                1           0  
1          7.333333                3           0  
2          8.250000    

In [17]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder

df = features_df[features_df["gsc_impressions"] > 0].copy()
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"]

df = df.dropna(subset=["word_count", "search_volume", "gsc_avg_position"])
print(f"Rows after dropping missing values: {len(df):,}")
# position buckets + impression-weighted benchmark (same approach as the capstone paper)
bins = [0, 3, 5, 10, 20, 50, np.inf]
labels = ["1-3", "4-5", "6-10", "11-20", "21-50", "51+"]
df["position_bucket"] = pd.cut(df["gsc_avg_position"], bins=bins, labels=labels)

bench = df.groupby("position_bucket", observed=True).apply(
    lambda g: g["gsc_clicks"].sum() / g["gsc_impressions"].sum()
)
df["expected_ctr"] = df["position_bucket"].map(bench).astype(float)
df["is_opportunity"] = (df["ctr"] < df["expected_ctr"]).astype(int)

print("Proxy label balance:")
print(df["is_opportunity"].value_counts(normalize=True).round(3))

# --- HONEST quick score: only the 5 declared features ---
honest_features = ["content_type", "main_intent", "word_count", "search_volume", "gsc_avg_position"]
X_honest = pd.get_dummies(df[honest_features], columns=["content_type", "main_intent"])
y = df["is_opportunity"]

Xh_tr, Xh_te, y_tr, y_te = train_test_split(X_honest, y, test_size=0.25, random_state=42)
model_honest = LogisticRegression(max_iter=2000).fit(Xh_tr, y_tr)
honest_acc = accuracy_score(y_te, model_honest.predict(Xh_te))
print(f"\nHonest quick score (5 features only): {honest_acc:.3f}")

# --- THE TRAP: deliberately add gsc_clicks — a raw ingredient of the label itself ---
X_leak = X_honest.copy()
X_leak["gsc_clicks"] = df["gsc_clicks"].values

Xl_tr, Xl_te, _, _ = train_test_split(X_leak, y, test_size=0.25, random_state=42)
model_leak = LogisticRegression(max_iter=2000).fit(Xl_tr, y_tr)
leak_acc = accuracy_score(y_te, model_leak.predict(Xl_te))
print(f"Leaked quick score (gsc_clicks added):  {leak_acc:.3f}")

# --- delete the leaked column, keep the honest number ---
del X_leak
print(f"\nLeak removed. Honest number stands: {honest_acc:.3f}")

Rows after dropping missing values: 2,311,532


/tmp/ipykernel_1982/4191851845.py:18: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  bench = df.groupby("position_bucket", observed=True).apply(


Proxy label balance:
is_opportunity
1    0.837
0    0.163
Name: proportion, dtype: float64

Honest quick score (5 features only): 0.837
Leaked quick score (gsc_clicks added):  0.932

Leak removed. Honest number stands: 0.837


Named limitation: this contract's proxy label is highly imbalanced (83.7% "opportunity" /
16.3% "not"), and the honest 5-feature model exactly matches that base rate (0.837) — meaning
these five features currently provide zero measurable lift over guessing the majority class.
This is a real, honest finding about this feature set on this month's slice, not a data
quality problem — content_type, main_intent, word_count, search_volume, and gsc_avg_position
alone don't yet distinguish opportunity pages from non-opportunity pages. A stronger feature
set (e.g. content freshness, query-level signals from fact_query_90d) may be needed before
this proxy is genuinely rankable — that's future work, not something this contract claims to
have solved.

Also: only 36.7% of March rows had gsc_data_available = TRUE (Section 3) — this contract's
9.8M-row table shrank to 2.3M analyzable rows after availability + missing-value filtering,
meaning almost 3 in 4 raw rows in this

Five features (max), each knowable at the decision moment — before any click/CTR outcome
for that day is known:

1. content_type — knowable: static content attribute, set when the page was created, not
   affected by that day's performance.
2. main_intent — knowable: same, a static content classification.
3. word_count — knowable: fixed content property, doesn't change based on daily traffic.
4. search_volume — knowable: an external market signal (how often the keyword is searched),
   independent of how this specific page performs.
5. gsc_avg_position — knowable: position is assigned by the search engine before any click
   happens that day; it's the cause of CTR, not a result of it.

Not included as features (pulled into the frame only to build the proxy/label downstream):
gsc_impressions, gsc_clicks — these are the raw ingredients CTR is computed from, and my
proxy label is built on CTR. Section 3's trap exercise below demonstrates exactly why.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous hashes used)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.